In [2]:
from pathlib import Path

student_files = sorted(Path("data").glob("student_text_*.txt"))

student_texts = []
for file in student_files:
    with open(file, "r", encoding="utf-8") as f:
        student_texts.append({
            "filename": file.stem,
            "text": f.read()
        })

student_texts[0]["filename"]

'student_text_1'

In [10]:
from typing import Literal
from pydantic import BaseModel, Field

class GradeAssessment(BaseModel):
    proposed_grade: Literal["A", "B", "C", "D", "E", "F"] = Field(
        description="Proposed Swedish grade"
    )
    motivation: str = Field(
        description="Short but concrete explanation for the proposed grade"
    )
    improvements: list[str] = Field(
        description="Concrete suggestions for how the student can improve"
    )

In [13]:
from pydantic_ai import Agent

grading_agent = Agent(
    "openrouter:nvidia/nemotron-3-super-120b-a12b:free",
    system_prompt="""
Du är en hjälpassistent för en svensklärare.

Din uppgift är att bedöma en elevtext och föreslå ett betyg.
var försiktig, konkret och pedagogisk. 
Hitta inte på saker som inte finns i texten.
Motivera utifrån textens kvalitet.

Returnerna:
- proposed_grade
- motivation
- improvements

Regler: 
- proposed_grade måste vara ett av: A, b, C, D, E, F
- motivation ska vara tydlig och kortfattad
- improvements ska vara konkreta och förbättringsförslag
- svara på svenska
"""
)

async def grade_student_text(student_text: str) -> GradeAssessment:
    result = await grading_agent.run(
        student_text,
        output_type=GradeAssessment
    )
    return result.output

In [14]:
assessments = []

for item in student_texts:
    assessment = await grade_student_text(item["text"])
    assessments.append({
        "filename": item["filename"],
        "assessment": assessment
    })

assessments[0]

{'filename': 'student_text_1',
 'assessment': GradeAssessment(proposed_grade='E', motivation='Texten visar förståelse för ämnet och använder en källa samt personliga reflektioner, men språket innehåller många stavfel, grammatiska fel och saknar sammanhang, vilket gör läsningen svår och påverkar tydligheten.', improvements=['Rätta stavfel exempelvis "andledningarna" → "anledningarna", "tal rädd" → "talarädd" eller "rädd för att tala".', 'Använd rätt punkter och kommatecken för att undvika långa meningar och förbättra läsbarheten.', 'Variera ordval och undvik upprepningar (t.ex. "rädd" flera gånger i rad).', 'Förbättra referensen till Peter Letmark: ange artikelens titel och korrekt datumformat.', 'Utveckla resonemanget med fler exempel och koppla tydligt till källan.', 'Arbeta med textens disposition: tydlig inledning, argument och avslutning.'])}

In [15]:
output_root = Path("grading_outputs")
output_root.mkdir(exist_ok=True)

for item in assessments:
    student_folder = output_root / item["filename"]
    student_folder.mkdir(exist_ok=True)

    assessment = item["assessment"]

    with open(student_folder / "proposed_grade.txt", "w", encoding="utf-8") as f:
        f.write(assessment.proposed_grade)

    with open(student_folder / "motivation.txt", "w", encoding="utf-8") as f:
        f.write(assessment.motivation)

    with open(student_folder / "improvements.txt", "w", encoding="utf-8") as f:
        f.write("\n".join(f"- {imp}" for imp in assessment.improvements))